## Table of Contents

1. [Imports](#Imports)
2. [Loading Datasets](#Loading-the-Datasets)

3. [Clean Books, User & Rating Datasets](#Cleaning-Books,-Users-&-Ratings-Data)
    - [Books Dataset](#BOOKS-DATASET)
    - [User Dataset](#USER-DATASET)
    - [Rating Dataset](#RATING-DATASET)

4. [Clean Review Datasets](#Cleaning-Review-Datasets)
    - [Customer Review Dataset](#CUSTOMER-REVIEW-DATASET)
    - [All Review Dataset](#ALL_REVIEW-DATASET)
    - [Book Review Dataset](#BOOK-REVIEW-DATASET)

5. [Merging Review Datasets](#Merge-All-Cleaned-Datasets)

6. [Text Preprocessing](#Text-Preprocessing-(for-NLP))


## Imports

In [1]:
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Downloading NLTK resources
#nltk.download('punkt')
#nltk.download('stopwords')

## Loading the Datasets

In [2]:
#Loading Book dataset
book_df=pd.read_csv(r"C:\Users\HP\Desktop\Bookify\Database\Datasets\Books.csv")

#Loading user dataset
user_df=pd.read_csv(r"C:\Users\HP\Desktop\Bookify\Database\Datasets\Users.csv")

#Loading Rating dataset
rating_df=pd.read_csv(r"C:\Users\HP\Desktop\Bookify\Database\Datasets\Ratings.csv")

#Loading customer review dataset
cust_review_df=pd.read_csv(r"C:\Users\HP\Desktop\Bookify\Database\Datasets\Customer_Reviews.csv")

#Loading all review dataset
all_review_df=pd.read_csv(r"C:\Users\HP\Desktop\Bookify\Database\Datasets\all_review .csv")

#Loading book review dataset
book_review_df = pd.read_csv(r"C:\Users\HP\Desktop\Bookify\Database\Datasets\book_review.csv", encoding='utf-8', engine='python').applymap(lambda x: ''.join([i if ord(i) < 128 else '' for i in str(x)]) if isinstance(x, str) else x)


# Cleaning Books, Users & Ratings Data

### BOOKS DATASET

In [3]:
book_df.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [4]:
book_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271360 entries, 0 to 271359
Data columns (total 8 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   ISBN                 271360 non-null  object
 1   Book-Title           271360 non-null  object
 2   Book-Author          271359 non-null  object
 3   Year-Of-Publication  271360 non-null  object
 4   Publisher            271358 non-null  object
 5   Image-URL-S          271360 non-null  object
 6   Image-URL-M          271360 non-null  object
 7   Image-URL-L          271357 non-null  object
dtypes: object(8)
memory usage: 16.6+ MB


In [5]:
book_df.describe()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
count,271360,271360,271359,271360,271358,271360,271360,271357
unique,271360,242135,102023,202,16807,271044,271044,271041
top,0195153448,Selected Poems,Agatha Christie,2002,Harlequin,http://images.amazon.com/images/P/185326119X.0...,http://images.amazon.com/images/P/185326119X.0...,http://images.amazon.com/images/P/225307649X.0...
freq,1,27,632,13903,7535,2,2,2


In [6]:
book_df.columns

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-S', 'Image-URL-M', 'Image-URL-L'],
      dtype='object')

In [7]:
book_df.shape

(271360, 8)

### 1. Remove Duplicates

In [8]:
#identifying the duplicates
book_df.duplicated().sum()

0

### 2. Handle Missing Values

In [9]:
#Finding the Missing values
book_df.isnull().sum()

ISBN                   0
Book-Title             0
Book-Author            1
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64

In [10]:
# Filling missing Book author values with 'Unknown'
book_df['Book-Author'].fillna('Unknown', inplace=True)

# Filling missing Publisher values with 'Unknown'
book_df['Publisher'].fillna('Unknown', inplace=True)

# Filling missing large image URLs values with 'Unknown'
book_df['Image-URL-L'].fillna('Unknown', inplace=True)

In [11]:
book_df.isnull().sum()

ISBN                   0
Book-Title             0
Book-Author            0
Year-Of-Publication    0
Publisher              0
Image-URL-S            0
Image-URL-M            0
Image-URL-L            0
dtype: int64

### 3. Standardize Book Publication Years

In [12]:
# Convert year column to numeric, handle invalid entries
book_df['Year-Of-Publication'] = pd.to_numeric(book_df['Year-Of-Publication'], errors='coerce')

# Replace years outside 1800–2025 range with NaN
book_df.loc[(book_df['Year-Of-Publication'] < 1800) | (book_df['Year-Of-Publication'] > 2025), 'Year-Of-Publication'] = None

# Fill missing years with median of valid years
year = int(book_df['Year-Of-Publication'].median())
book_df['Year-Of-Publication'].fillna(year, inplace=True)

# Convert to integer
book_df['Year-Of-Publication'] = book_df['Year-Of-Publication'].astype(int)

In [13]:
#Saving the Cleaned Book Dataset
book_df.to_csv(r'C:\Users\HP\Desktop\Bookify\Database\Cleaned_Datasets\Books_Cleaned.csv', index=False)

### USER DATASET

In [14]:
user_df.head()

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


In [15]:
user_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 278858 entries, 0 to 278857
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   User-ID   278858 non-null  int64  
 1   Location  278858 non-null  object 
 2   Age       168096 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 6.4+ MB


In [16]:
user_df.describe()

,User-ID,Age
count,278858.00000,168096.000000
mean,139429.50000,34.751434
std,80499.51502,14.428097
min,1.00000,0.000000
25%,69715.25000,24.000000
50%,139429.50000,32.000000
75%,209143.75000,44.000000
max,278858.00000,244.000000


In [17]:
user_df.shape

(278858, 3)

### 1. Remove Duplicates

In [18]:
#identifying the duplicates
user_df.duplicated().sum()

0

### 2. Handle Missing Values

In [19]:
#Finding the Missing values
user_df.isnull().sum()

User-ID          0
Location         0
Age         110762
dtype: int64

In [20]:
# Filter rows with 'Age' missing or between 5 and 100.
user_df = user_df[(user_df['Age'].isnull()) | ((user_df['Age'] >= 5) & (user_df['Age'] <= 100))]

# Fill missing 'Age' values with the median.
user_df['Age'].fillna(user_df['Age'].median(), inplace=True)

In [21]:
#Finding the Missing values
user_df.isnull().sum()

User-ID     0
Location    0
Age         0
dtype: int64

In [22]:
#Saving the Cleaned Users Dataset
user_df.to_csv(r'Cleaned_Datasets\Users_Cleaned.csv', index=False)

### RATING DATASET

In [23]:
rating_df.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [24]:
rating_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1149780 entries, 0 to 1149779
Data columns (total 3 columns):
 #   Column       Non-Null Count    Dtype 
---  ------       --------------    ----- 
 0   User-ID      1149780 non-null  int64 
 1   ISBN         1149780 non-null  object
 2   Book-Rating  1149780 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 26.3+ MB


In [25]:
rating_df.describe()

,User-ID,Book-Rating
count,1.149780e+06,1.149780e+06
mean,1.403864e+05,2.866950e+00
std,8.056228e+04,3.854184e+00
min,2.000000e+00,0.000000e+00
25%,7.034500e+04,0.000000e+00
50%,1.410100e+05,0.000000e+00
75%,2.110280e+05,7.000000e+00
max,2.788540e+05,1.000000e+01


In [26]:
rating_df.shape

(1149780, 3)

### 1. Remove Duplicates

In [27]:
#identifying the duplicates
rating_df.duplicated().sum()

0

### 2. Handle Missing Values

In [28]:
#Finding the Missing values
rating_df.isnull().sum()

User-ID        0
ISBN           0
Book-Rating    0
dtype: int64

In [29]:
#Saving the Cleaned Rating Dataset
rating_df.to_csv(r'C:\Users\HP\Desktop\Bookify\Database\Cleaned_Datasets\Ratings_Cleaned.csv', index=False)

# Cleaning Review Datasets

### Clean Special Characters

In [30]:
# Function to clean text by removing special characters, extra spaces, etc.
def clean_text(text):
    if isinstance(text, str):
        # Remove any non-ASCII characters and special characters
        text = re.sub(r'[^\x00-\x7F]+', '', text)  # Remove non-ASCII characters
        text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation and special characters
        text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

### CUSTOMER REVIEW DATASET

In [31]:
cust_review_df.head()

,Unnamed: 0,book name,review title,reviewer,reviewer rating,review description,is_verified,date,timestamp,ASIN,Author
0,17,"Friends, Lovers, and the Big Terrible Thing: A...",A very sad read,Veronica R Ewing,4,What a shock to lose such a talented and funny...,True,30-10-2023,"Reviewed in the United States October 30, 2023",1250866448,Matthew Perry
1,131,Lessons in Chemistry: A Novel,I LOVE THIS BOOK!! 😍 ⭐️⭐️⭐️⭐️⭐️,Sonia,5,"Oh, my God!! I LOVE THIS BOOK SO, SO, SO MUCH!...",True,24-10-2023,"Reviewed in the United States October 24, 2023",038554734X,Bonnie Garmus
2,464,Flash Cards: Sight Words,Amazing for struggling readers,Ryan Williams,5,I bought these for my son who was struggling r...,True,29-09-2023,"Reviewed in the United States September 29, 2023",1338233580,Scholastic
3,644,A Court of Mist and Fury (A Court of Thorns an...,"The ending was stunning, as always, but I had ...",Brittany,4,** Warning: This is NOT a spoiler-free review ...,True,29-06-2016,"Reviewed in the United States June 29, 2016",1635575583,Sarah J. Maas
4,78,The Ballad of Songbirds and Snakes (A Hunger G...,So Good!!!,Kindle Customer,5,"If you loved the Hunger Games, you have to rea...",True,29-10-2023,"Reviewed in the United States October 29, 2023",1339016575,Suzanne Collins


In [32]:
cust_review_df.describe()

,Unnamed: 0,reviewer rating
count,910.00000,910.000000
mean,454.50000,4.827473
std,262.83867,0.432346
min,0.00000,2.000000
25%,227.25000,5.000000
50%,454.50000,5.000000
75%,681.75000,5.000000
max,909.00000,5.000000


In [33]:
cust_review_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 910 entries, 0 to 909
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Unnamed: 0          910 non-null    int64 
 1   book name           910 non-null    object
 2   review title        910 non-null    object
 3   reviewer            910 non-null    object
 4   reviewer rating     910 non-null    int64 
 5   review description  910 non-null    object
 6   is_verified         910 non-null    bool  
 7   date                910 non-null    object
 8   timestamp           910 non-null    object
 9   ASIN                910 non-null    object
 10  Author              910 non-null    object
dtypes: bool(1), int64(2), object(8)
memory usage: 72.1+ KB


In [34]:
cust_review_df.shape

(910, 11)

### 1. Cleaning and Standardizing Dataset

In [35]:
#Extracting relevant columns and standardizing columns
cust_review_cleaned = cust_review_df[["ASIN", "reviewer", "review description", "reviewer rating", "date"]].copy()
cust_review_cleaned.rename(columns={
    "ASIN": "book_id",
    "reviewer": "user_id",
    "review description": "review_text",
    "reviewer rating": "rating",
    "date": "review_date"
}, inplace=True)
cust_review_cleaned["source"] = "Customer"
cust_review_cleaned=cust_review_cleaned[["book_id", "user_id", "review_text", "rating", "review_date", "source"]]


In [36]:
cust_review_cleaned.head()

,book_id,user_id,review_text,rating,review_date,source
0,1250866448,Veronica R Ewing,What a shock to lose such a talented and funny...,4,30-10-2023,Customer
1,038554734X,Sonia,"Oh, my God!! I LOVE THIS BOOK SO, SO, SO MUCH!...",5,24-10-2023,Customer
2,1338233580,Ryan Williams,I bought these for my son who was struggling r...,5,29-09-2023,Customer
3,1635575583,Brittany,** Warning: This is NOT a spoiler-free review ...,4,29-06-2016,Customer
4,1339016575,Kindle Customer,"If you loved the Hunger Games, you have to rea...",5,29-10-2023,Customer


In [37]:
cust_review_cleaned["review_date"] = pd.to_datetime(cust_review_cleaned["review_date"])

In [38]:
cust_review_cleaned.dtypes

book_id                object
user_id                object
review_text            object
rating                  int64
review_date    datetime64[ns]
source                 object
dtype: object

In [39]:
#Saving the Cleaned customer review Dataset
cust_review_cleaned.to_csv(r'Cleaned_Datasets\Customer_Review_Cleaned.csv',index=False)

### ALL_REVIEW DATASET

In [40]:
all_review_df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [41]:
all_review_df.describe()

,Unnamed: 0.1,Unnamed: 0,rating,unixReviewTime
count,12000.00000,12000.000000,12000.000000,1.200000e+04
mean,5999.50000,10024.275667,3.250000,1.344537e+09
std,3464.24595,10502.233123,1.421619,4.369374e+07
min,0.00000,0.000000,1.000000,9.602496e+08
25%,2999.75000,2999.750000,2.000000,1.316218e+09
50%,5999.50000,5999.500000,3.500000,1.356826e+09
75%,8999.25000,12475.750000,4.250000,1.376870e+09
max,11999.00000,47770.000000,5.000000,1.405814e+09


In [42]:
all_review_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Unnamed: 0.1    12000 non-null  int64 
 1   Unnamed: 0      12000 non-null  int64 
 2   asin            12000 non-null  object
 3   helpful         12000 non-null  object
 4   rating          12000 non-null  int64 
 5   reviewText      12000 non-null  object
 6   reviewTime      12000 non-null  object
 7   reviewerID      12000 non-null  object
 8   reviewerName    11962 non-null  object
 9   summary         12000 non-null  object
 10  unixReviewTime  12000 non-null  int64 
dtypes: int64(4), object(7)
memory usage: 1.0+ MB


In [43]:
all_review_df.shape

(12000, 11)

### 1. Cleaning and Standardizing Dataset

In [44]:
#Extracting relevant columns and standardizing columns
all_review_cleaned= all_review_df[["asin", "reviewerID", "reviewText", "rating", "reviewTime"]].copy()
all_review_cleaned.rename(columns={
    "asin": "book_id",
    "reviewerID": "user_id",
    "reviewText": "review_text",
    "rating": "rating",
    "reviewTime": "review_date"
}, inplace=True)
all_review_cleaned["source"] = "Amazon"

In [45]:
all_review_cleaned.head()

,book_id,user_id,review_text,rating,review_date,source
0,B0033UV8HI,A3HHXRELK8BHQG,"Jace Rankin may be short, but he's nothing to ...",3,"09 2, 2010",Amazon
1,B002HJV4DE,A2RGNZ0TRF578I,Great short read. I didn't want to put it dow...,5,"10 8, 2013",Amazon
2,B002ZG96I4,A3S0H2HV6U1I7F,I'll start by saying this is the first of four...,3,"04 11, 2014",Amazon
3,B002QHWOEU,AC4OQW3GZ919J,Aggie is Angela Lansbury who carries pocketboo...,3,"07 5, 2014",Amazon
4,B001A06VJ8,A3C9V987IQHOQD,I did not expect this type of book to be in li...,4,"12 31, 2012",Amazon


In [46]:
all_review_cleaned["review_date"] = pd.to_datetime(all_review_cleaned["review_date"])

In [47]:
all_review_cleaned.dtypes

book_id                object
user_id                object
review_text            object
rating                  int64
review_date    datetime64[ns]
source                 object
dtype: object

In [48]:
# Apply cleaning function to textual columns 
all_review_cleaned['review_text'] = all_review_cleaned['review_text'].apply(clean_text)

In [49]:
#Saving the Cleaned all review Dataset
all_review_cleaned.to_csv(r'Cleaned_Datasets\All_Review_Cleaned.csv',index=False)

### BOOK REVIEW DATASET

In [50]:
book_review_df.head()

,ratings,reviews
0,5,"When Jennifer, a college student, returns to h..."
1,5,Growing up in the Appalachian mountains of sou...
2,5,"The heat is lifting some now, so a little wind..."
3,5,"When I was growing up, all family history was ..."
4,5,This is a story about five generations of the ...


In [51]:
book_review_df.describe()

,ratings
count,66592.000000
mean,3.920741
std,1.250043
min,1.000000
25%,3.000000
50%,4.000000
75%,5.000000
max,5.000000


In [52]:
book_review_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66592 entries, 0 to 66591
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   ratings  66592 non-null  int64 
 1   reviews  66592 non-null  object
dtypes: int64(1), object(1)
memory usage: 1.0+ MB


In [53]:
book_review_df.shape

(66592, 2)

### 1. Cleaning and Standardizing Dataset

In [54]:
#Extracting relevant columns and standardizing columns
book_review_cleaned = book_review_df[["ratings", "reviews"]].copy()
book_review_cleaned.rename(columns={
    "ratings": "rating",
    "reviews": "review_text"
}, inplace=True)

book_review_cleaned["user_id"] = "unknown"
book_review_cleaned["book_id"] = "unknown"
book_review_cleaned["review_date"] =pd.NaT
book_review_cleaned["source"] = "Goodreads"

In [55]:
book_review_cleaned.head()

,rating,review_text,user_id,book_id,review_date,source
0,5,"When Jennifer, a college student, returns to h...",unknown,unknown,NaT,Goodreads
1,5,Growing up in the Appalachian mountains of sou...,unknown,unknown,NaT,Goodreads
2,5,"The heat is lifting some now, so a little wind...",unknown,unknown,NaT,Goodreads
3,5,"When I was growing up, all family history was ...",unknown,unknown,NaT,Goodreads
4,5,This is a story about five generations of the ...,unknown,unknown,NaT,Goodreads


In [56]:
book_review_cleaned.shape

(66592, 6)

In [57]:
# Apply cleaning function to textual columns 
book_review_cleaned['review_text'] = book_review_cleaned['review_text'].apply(clean_text)

In [58]:
#Saving the Cleaned Book review Dataset
book_review_cleaned.to_csv(r'Cleaned_Datasets\Book_Review_Cleaned.csv',index=False)

# Merge All Cleaned Datasets 

In [59]:
#Merging datasets
merged_reviews = pd.concat([all_review_cleaned, book_review_cleaned,cust_review_cleaned], ignore_index=True)

In [60]:
#finding duplictates
merged_reviews.duplicated().sum()

2349

In [61]:
# Remove exact duplicate rows
merged_reviews.drop_duplicates(inplace=True)

In [62]:
merged_reviews.duplicated().sum()

0

In [63]:
# Add a unique ID to each review
merged_reviews.reset_index(drop=True, inplace=True)
merged_reviews.insert(0, 'review_id', merged_reviews.index + 1)

In [64]:
# Check the merged dataframe
merged_reviews.head()

,review_id,book_id,user_id,review_text,rating,review_date,source
0,1,B0033UV8HI,A3HHXRELK8BHQG,Jace Rankin may be short but hes nothing to me...,3,2010-09-02,Amazon
1,2,B002HJV4DE,A2RGNZ0TRF578I,Great short read I didnt want to put it down s...,5,2013-10-08,Amazon
2,3,B002ZG96I4,A3S0H2HV6U1I7F,Ill start by saying this is the first of four ...,3,2014-04-11,Amazon
3,4,B002QHWOEU,AC4OQW3GZ919J,Aggie is Angela Lansbury who carries pocketboo...,3,2014-07-05,Amazon
4,5,B001A06VJ8,A3C9V987IQHOQD,I did not expect this type of book to be in li...,4,2012-12-31,Amazon


In [65]:
print("Shape of merged dataset:", merged_reviews.shape)

Shape of merged dataset: (77153, 7)


In [66]:
#Checking for Missing values
merged_reviews.isnull().sum()

review_id          0
book_id            0
user_id            0
review_text        0
rating             0
review_date    64243
source             0
dtype: int64

In [67]:
merged_reviews.dtypes

review_id               int64
book_id                object
user_id                object
review_text            object
rating                  int64
review_date    datetime64[ns]
source                 object
dtype: object

In [68]:
#setting default date
merged_reviews['review_date'].fillna(pd.to_datetime('2025-01-01'), inplace=True)

In [69]:
#Saving the Cleaned Merged review Dataset
merged_reviews.to_csv(r'Cleaned_Datasets\Merge_Review_Cleaned.csv',index=False)

# Text Preprocessing (for NLP)

In [70]:
review_df=pd.read_csv(r"Cleaned_Datasets\Merge_Review_Cleaned.csv")

In [71]:
review_df.head()

,review_id,book_id,user_id,review_text,rating,review_date,source
0,1,B0033UV8HI,A3HHXRELK8BHQG,Jace Rankin may be short but hes nothing to me...,3,2010-09-02,Amazon
1,2,B002HJV4DE,A2RGNZ0TRF578I,Great short read I didnt want to put it down s...,5,2013-10-08,Amazon
2,3,B002ZG96I4,A3S0H2HV6U1I7F,Ill start by saying this is the first of four ...,3,2014-04-11,Amazon
3,4,B002QHWOEU,AC4OQW3GZ919J,Aggie is Angela Lansbury who carries pocketboo...,3,2014-07-05,Amazon
4,5,B001A06VJ8,A3C9V987IQHOQD,I did not expect this type of book to be in li...,4,2012-12-31,Amazon


In [72]:
review_df.shape

(77153, 7)

In [73]:
review_df.isnull().sum()

review_id      0
book_id        0
user_id        0
review_text    5
rating         0
review_date    0
source         0
dtype: int64

In [74]:
#Filling the row having NaN values with No Review
review_df['review_text'] = review_df['review_text'].fillna('No Review')

In [75]:
def clean_review(text):
    if pd.isnull(text):
        return ""
    
    # Convert text to lowercase(Case Normalization)
    text = text.lower()
    
    # Special Character Removal
    text = re.sub(r'[^a-z0-9\s]', '', text)

    # Tokenization
    tokens = word_tokenize(text)

    # Stopword Removal 
    stop_words = set(stopwords.words('english'))
    custom_stopwords = {"book","story","surprise","learn","novel","author"}  #read,love
    stop_words.update(custom_stopwords)

    filtered_tokens = [word for word in tokens if word not in stop_words]

    # return token as string
    return ' '.join(filtered_tokens) 


In [76]:
# Applying clean_review to review text column
review_df['cleaned_tokens'] = review_df['review_text'].apply(clean_review)

In [77]:
# Check for empty rows after stop word removal and fill with No Review
review_df['cleaned_tokens'] = review_df['cleaned_tokens'].replace("", "No Review")

In [78]:
review_df.head()

,review_id,book_id,user_id,review_text,rating,review_date,source,cleaned_tokens
0,1,B0033UV8HI,A3HHXRELK8BHQG,Jace Rankin may be short but hes nothing to me...,3,2010-09-02,Amazon,jace rankin may short hes nothing mess man hau...
1,2,B002HJV4DE,A2RGNZ0TRF578I,Great short read I didnt want to put it down s...,5,2013-10-08,Amazon,great short read didnt want put read one sitti...
2,3,B002ZG96I4,A3S0H2HV6U1I7F,Ill start by saying this is the first of four ...,3,2014-04-11,Amazon,ill start saying first four books wasnt expect...
3,4,B002QHWOEU,AC4OQW3GZ919J,Aggie is Angela Lansbury who carries pocketboo...,3,2014-07-05,Amazon,aggie angela lansbury carries pocketbooks inst...
4,5,B001A06VJ8,A3C9V987IQHOQD,I did not expect this type of book to be in li...,4,2012-12-31,Amazon,expect type library pleased find price right


In [79]:
review_df.duplicated().sum()

0

In [80]:
review_df.isnull().sum()

review_id         0
book_id           0
user_id           0
review_text       0
rating            0
review_date       0
source            0
cleaned_tokens    0
dtype: int64

In [81]:
review_df.to_csv(r"Cleaned_Datasets\preprocessed_review.csv",index=False)